In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('../experiments/results.csv', parse_dates=['timestamp'])

SCORE_COLS = ['overall_score', 'structure', 'data_richness', 'sophistication', 'actionability', 'sentiment_balance']
SCORE_COLS = [c for c in SCORE_COLS if c in df.columns]

df.shape, df['instrument'].unique(), df['mode'].unique()

((104, 15),
 <ArrowStringArray>
 ['AAPL', 'MSFT', 'JPM', 'BAC', 'JNJ', 'PFE', 'XOM', 'WMT', 'KO', 'CAT']
 Length: 10, dtype: str,
 <ArrowStringArray>
 ['sequential', 'group_chat']
 Length: 2, dtype: str)

In [2]:
# Mean ± std overall_score per instrument
agg = df.groupby('instrument')['overall_score'].agg(['mean', 'std', 'count']).reset_index()
agg = agg.sort_values('mean', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=agg['instrument'], y=agg['mean'],
    error_y=dict(type='data', array=agg['std'].fillna(0)),
    marker_color=agg['mean'],
    marker_colorscale='RdYlGn',
    text=agg['mean'].round(1), textposition='outside'
))
fig.update_layout(title='Overall Score — mean ± std per instrument', yaxis_range=[0, 105], height=450)
fig.show()

In [ ]:
fig = px.box(
    df.sort_values('instrument'), x='instrument', y='overall_score',
    color='instrument', points='all',
    title='Overall Score distribution per instrument'
)
fig.update_layout(showlegend=False, height=450)
fig.show()

In [ ]:
agg_mode = df.groupby('mode')['overall_score'].agg(['mean', 'std', 'count']).reset_index()

fig = px.bar(
    agg_mode, x='mode', y='mean', error_y='std',
    text='mean', color='mean', color_continuous_scale='RdYlGn',
    title='Overall Score — mean ± std per mode'
)
fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.update_layout(yaxis_range=[0, 105], height=400, coloraxis_showscale=False)
fig.show()

In [5]:
# Heatmap: instrument × mode (mean overall_score)
if df['mode'].nunique() > 1:
    pivot = df.groupby(['instrument', 'mode'])['overall_score'].mean().unstack(fill_value=None)
    fig = px.imshow(
        pivot, text_auto='.1f', color_continuous_scale='RdYlGn',
        title='Mean Overall Score — instrument × mode', aspect='auto'
    )
    fig.update_layout(height=max(300, len(pivot) * 40))
    fig.show()
else:
    print('Only one mode so far — heatmap will show once group_chat data arrives.')

In [6]:
# Sub-metric radar per mode (mean)
radar_cols = [c for c in ['structure', 'data_richness', 'sophistication', 'actionability', 'sentiment_balance'] if c in df.columns]
radar_df = df.groupby('mode')[radar_cols].mean().reset_index()

fig = go.Figure()
for _, row in radar_df.iterrows():
    vals = list(row[radar_cols]) + [row[radar_cols[0]]]
    cats = radar_cols + [radar_cols[0]]
    fig.add_trace(go.Scatterpolar(r=vals, theta=cats, fill='toself', name=row['mode']))
fig.update_layout(polar=dict(radialaxis=dict(range=[0, 100])), title='Sub-metric radar per mode', height=500)
fig.show()

In [7]:
# Sub-metric heatmap: instrument × metric
metric_pivot = df.groupby('instrument')[radar_cols].mean()
metric_pivot = metric_pivot.loc[df.groupby('instrument')['overall_score'].mean().sort_values(ascending=False).index]

fig = px.imshow(
    metric_pivot, text_auto='.1f', color_continuous_scale='RdYlGn',
    title='Mean sub-metrics per instrument', aspect='auto'
)
fig.update_layout(height=max(300, len(metric_pivot) * 40 + 100))
fig.show()

In [8]:
# execution_time per instrument
fig = px.box(
    df.sort_values('instrument'), x='instrument', y='execution_time',
    color='mode', points='all',
    title='Execution time (s) per instrument'
)
fig.update_layout(height=450)
fig.show()

In [9]:
# Grade distribution
if 'grade' in df.columns:
    grade_order = ['A+', 'A', 'A-', 'B+', 'B', 'B-', 'C+', 'C', 'C-', 'D', 'F']
    grade_counts = df.groupby(['mode', 'grade']).size().reset_index(name='count')
    fig = px.bar(
        grade_counts, x='grade', y='count', color='mode',
        barmode='group', category_orders={'grade': grade_order},
        title='Grade distribution per mode'
    )
    fig.update_layout(height=400)
    fig.show()